In [ ]:
"""
Signature Verification Training — ArcFace + Contrastive Loss
=============================================================
Speed strategy:
  • ALL images (train + val + test) pre-loaded as float tensors into RAM once
  • __getitem__ = pure tensor dict lookup (no disk I/O, no PIL, no transform)
  • num_workers=0 — no forking, no pickle, no deadlock
  • GPU-side stochastic augmentation applied per-batch in train_epoch
  • non_blocking GPU transfers + autocast for max throughput
"""

import os, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image

from model import model_final
from dataset import (train_dataloader, test_dataloader, val_dataloader,
                     train_pairs, val_pairs, test_pairs, genuine_by_author)

# ─────────────────────────────────────────────────────────────────────────────
# DEVICE + SEEDS
# ─────────────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────────────────────────────────
# UNWRAP torch.compile
# ─────────────────────────────────────────────────────────────────────────────
def unwrap_model(m):
    if hasattr(m, '_orig_mod'): return m._orig_mod
    if hasattr(m, 'module'):    return m.module
    return m

raw_model = unwrap_model(model_final).to(device)

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def author_from_path(path):
    try:    return os.path.basename(path).split('_')[1]
    except: return "unknown"

# ─────────────────────────────────────────────────────────────────────────────
# BASE TRANSFORM  (deterministic — applied once at cache-build time)
# Matches test_transform: resize → grayscale → invert → tensor → normalize
# ─────────────────────────────────────────────────────────────────────────────
import torchvision.transforms as T

base_transform = T.Compose([
    T.Resize((64, 64)),
    T.Grayscale(num_output_channels=1),
    T.RandomInvert(p=1.0),          # always invert (p=1.0 is deterministic)
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,)),
])

# ─────────────────────────────────────────────────────────────────────────────
# GPU AUGMENTATION  (stochastic — applied per-batch on device tensors)
# Replaces CPU-side RandomAffine / RandomPerspective / GaussianBlur
# Input:  (B, 1, 64, 64) tensor on GPU, values in [-1, 1]
# Output: (B, 1, 64, 64) tensor on GPU, same range
# ─────────────────────────────────────────────────────────────────────────────
try:
    import kornia.augmentation as K
    _kornia_available = True
except ImportError:
    _kornia_available = False

if _kornia_available:
    gpu_augment = nn.Sequential(
        K.RandomAffine(degrees=15, translate=(0.08, 0.08), scale=(0.98, 1.1),
                       padding_mode='border', p=1.0),
        K.RandomPerspective(distortion_scale=0.2, p=0.25),
        K.RandomGaussianBlur((3, 3), sigma=(0.1, 0.5), p=0.3),
    ).to(device)
    print("  GPU augmentation: kornia ✓")
else:
    # Fallback: lightweight CPU tensor augmentation (still faster than PIL workers)
    gpu_augment = None
    print("  GPU augmentation: kornia not found — using light tensor augmentation")

def augment_batch(imgs):
    """Apply stochastic augmentation to a batch of images on device."""
    if _kornia_available and gpu_augment is not None:
        return gpu_augment(imgs)
    # Minimal fallback: random horizontal flip only
    if random.random() > 0.5:
        imgs = torch.flip(imgs, dims=[-1])
    return imgs


# ─────────────────────────────────────────────────────────────────────────────
# RAM TENSOR CACHE  (build once, reuse every epoch)
# ─────────────────────────────────────────────────────────────────────────────
def build_tensor_cache(all_pairs, desc="Caching"):
    """Load all unique images → base_transform → CPU tensor. Returns dict."""
    unique = list({p for pair in all_pairs for p in pair})
    cache  = {}
    for p in tqdm(unique, desc=f"  {desc}", leave=False):
        cache[p] = base_transform(Image.open(p).convert('L'))
    return cache


# ─────────────────────────────────────────────────────────────────────────────
# DATASET  (pure tensor lookup — __getitem__ has near-zero cost)
# ─────────────────────────────────────────────────────────────────────────────
class TensorPairDataset(Dataset):
    """
    All images pre-loaded as tensors in RAM.
    __getitem__ is a dict lookup + tensor copy — no disk I/O, no PIL, no transform.
    Fully compatible with num_workers=0.
    """
    def __init__(self, pairs, labels, author2id, tensor_cache):
        self.pairs        = pairs
        self.labels       = labels
        self.author2id    = author2id
        self.tensor_cache = tensor_cache

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        p1, p2 = self.pairs[idx]
        return (
            self.tensor_cache[p1],
            self.tensor_cache[p2],
            torch.tensor(self.labels[idx], dtype=torch.float32),
            torch.tensor(self.author2id.get(author_from_path(p1), 0), dtype=torch.long),
            torch.tensor(self.author2id.get(author_from_path(p2), 0), dtype=torch.long),
        )


# ─────────────────────────────────────────────────────────────────────────────
# ARCFACE LOSS
# ─────────────────────────────────────────────────────────────────────────────
class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, num_classes, s=32.0, m=0.50):
        super().__init__()
        self.s = s; self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels):
        W       = F.normalize(self.weight, p=2, dim=1)
        cosine  = F.linear(embeddings.float(), W.float())
        sine    = torch.sqrt((1.0 - cosine.pow(2)).clamp(1e-9, 1.0))
        phi     = cosine * self.cos_m - sine * self.sin_m
        phi     = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        logits  = (one_hot * phi + (1.0 - one_hot) * cosine) * self.s
        return F.cross_entropy(logits, labels)


# ─────────────────────────────────────────────────────────────────────────────
# CONTRASTIVE LOSS
# ─────────────────────────────────────────────────────────────────────────────
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, labels):
        d   = F.pairwise_distance(emb1.float(), emb2.float(), p=2)
        pos = labels       * d.pow(2)
        neg = (1 - labels) * F.relu(self.margin - d).pow(2)
        return (pos + neg).mean()


# ─────────────────────────────────────────────────────────────────────────────
# COMBINED LOSS
# ─────────────────────────────────────────────────────────────────────────────
class SignatureVerificationLoss(nn.Module):
    def __init__(self, num_classes, emb_dim=256, s=32.0, m=0.50,
                 margin=1.0, lambda_arc=0.6, lambda_con=0.4):
        super().__init__()
        self.arcface     = ArcFaceLoss(emb_dim, num_classes, s=s, m=m)
        self.contrastive = ContrastiveLoss(margin=margin)
        self.la = lambda_arc; self.lc = lambda_con

    def forward(self, emb1, emb2, pair_labels, auth1, auth2):
        arc   = (self.arcface(emb1, auth1) + self.arcface(emb2, auth2)) / 2.0
        con   = self.contrastive(emb1, emb2, pair_labels)
        total = self.la * arc + self.lc * con
        return total, arc.item(), con.item()


# ─────────────────────────────────────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def compute_distances(model, loader):
    model.eval()
    dists, labs = [], []
    for img1, img2, labels, _, _ in loader:
        e1, e2 = model(img1.to(device, non_blocking=True),
                       img2.to(device, non_blocking=True))
        dists.append(F.pairwise_distance(e1.float(), e2.float()).cpu().numpy())
        labs.append(labels.numpy())
    return np.concatenate(dists), np.concatenate(labs)


def best_threshold(distances, labels):
    best_acc, best_t = 0.0, 0.5
    for t in np.linspace(distances.min(), distances.max(), 200):
        acc = accuracy_score(labels, (distances < t).astype(int))
        if acc > best_acc: best_acc, best_t = acc, t
    return best_t, best_acc


def evaluate(model, loader, split="Val"):
    dists, labels = compute_distances(model, loader)
    t, acc = best_threshold(dists, labels)
    try:    auc = roc_auc_score(labels, -dists)
    except: auc = 0.5
    preds   = (dists < t).astype(int)
    genuine = labels == 1; forged = labels == 0
    far = float(np.mean(preds[forged]  == 1)) if forged.any()  else 0.0
    frr = float(np.mean(preds[genuine] == 0)) if genuine.any() else 0.0
    print(f"  [{split}] Acc={acc:.4f}  AUC={auc:.4f}  FAR={far:.4f}  FRR={frr:.4f}  Thresh={t:.4f}")
    return {"acc": acc, "auc": auc, "far": far, "frr": frr, "threshold": t}


# ─────────────────────────────────────────────────────────────────────────────
# TRAIN EPOCH  (GPU augmentation applied here)
# ─────────────────────────────────────────────────────────────────────────────
def train_epoch(model, criterion, optimizer, scheduler, scaler, loader, epoch):
    model.train()
    tot = tot_arc = tot_con = 0.0
    use_amp = (device.type == "cuda")
    pbar = tqdm(loader, desc=f"Epoch {epoch:03d} [Train]", leave=False)

    for img1, img2, pair_labels, auth1, auth2 in pbar:
        # Move to GPU
        img1        = img1.to(device, non_blocking=True)
        img2        = img2.to(device, non_blocking=True)
        pair_labels = pair_labels.to(device, non_blocking=True)
        auth1       = auth1.to(device, non_blocking=True)
        auth2       = auth2.to(device, non_blocking=True)

        # GPU-side stochastic augmentation (replaces CPU RandomAffine etc.)
        with torch.no_grad():
            img1 = augment_batch(img1)
            img2 = augment_batch(img2)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=use_amp):
            e1, e2         = model(img1, img2)
            loss, arc, con = criterion(e1, e2, pair_labels, auth1, auth2)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        tot += loss.item(); tot_arc += arc; tot_con += con
        pbar.set_postfix(loss=f"{loss.item():.4f}", arc=f"{arc:.4f}", con=f"{con:.4f}")

    n = len(loader)
    print(f"  [Train] Loss={tot/n:.4f}  ArcFace={tot_arc/n:.4f}  Contrastive={tot_con/n:.4f}")
    return tot / n


# ─────────────────────────────────────────────────────────────────────────────
# VAL EPOCH
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def val_epoch(model, criterion, loader):
    model.eval()
    tot = 0.0
    use_amp = (device.type == "cuda")
    for img1, img2, pair_labels, auth1, auth2 in loader:
        img1        = img1.to(device, non_blocking=True)
        img2        = img2.to(device, non_blocking=True)
        pair_labels = pair_labels.to(device, non_blocking=True)
        auth1       = auth1.to(device, non_blocking=True)
        auth2       = auth2.to(device, non_blocking=True)
        with autocast(enabled=use_amp):
            e1, e2     = model(img1, img2)
            loss, _, _ = criterion(e1, e2, pair_labels, auth1, auth2)
        tot += loss.item()
    avg = tot / len(loader)
    print(f"  [Val  ] Loss={avg:.4f}")
    return avg


# ─────────────────────────────────────────────────────────────────────────────
# MAIN TRAINING FUNCTION
# ─────────────────────────────────────────────────────────────────────────────
def train(
    model,
    train_dataloader, val_dataloader, test_dataloader,
    train_pairs, val_pairs, test_pairs,
    genuine_by_author,
    num_epochs=40, lr=3e-4, weight_decay=1e-4,
    arcface_margin=0.50, arcface_scale=32.0,
    contrastive_margin=1.0, lambda_arc=0.6, lambda_con=0.4,
    save_dir="./checkpoints", patience=10,
):
    os.makedirs(save_dir, exist_ok=True)

    # ── Author → id mapping ──────────────────────────────────────────────────
    author2id   = {a: i for i, a in enumerate(sorted(genuine_by_author.keys()))}
    num_classes = len(author2id)
    print(f"Signer classes (ArcFace): {num_classes}")

    # ── Kill original persistent workers ─────────────────────────────────────
    for ld in [train_dataloader, val_dataloader, test_dataloader]:
        try: ld._iterator = None
        except: pass

    # ── Labels from original datasets ────────────────────────────────────────
    train_labels = train_dataloader.dataset.labels
    val_labels   = val_dataloader.dataset.labels
    test_labels  = test_dataloader.dataset.labels

    # ── Build unified RAM tensor cache (all splits) ───────────────────────────
    print("\n  Building RAM tensor cache (all images → base transform → CPU tensor)...")
    all_pairs_combined = train_pairs + val_pairs + test_pairs
    tensor_cache = build_tensor_cache(all_pairs_combined, desc="Loading all images")
    n_mb = sum(t.nelement() * t.element_size() for t in tensor_cache.values()) / 1e6
    print(f"  Cached {len(tensor_cache)} unique images  ({n_mb:.0f} MB in RAM)")

    # ── Build datasets (all use same tensor cache) ────────────────────────────
    train_ds = TensorPairDataset(train_pairs, train_labels, author2id, tensor_cache)
    val_ds   = TensorPairDataset(val_pairs,   val_labels,   author2id, tensor_cache)
    test_ds  = TensorPairDataset(test_pairs,  test_labels,  author2id, tensor_cache)

    # num_workers=0: no forking, no pickle, no deadlock
    # Everything is in RAM — the bottleneck is GPU compute, not data loading
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,
                              num_workers=0, pin_memory=torch.cuda.is_available())
    val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False,
                              num_workers=0, pin_memory=torch.cuda.is_available())
    test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False,
                              num_workers=0, pin_memory=torch.cuda.is_available())

    print(f"  Loaders (num_workers=0, batch 128/512): "
          f"train={len(train_loader)} | val={len(val_loader)} | test={len(test_loader)} batches")

    # ── Loss ──────────────────────────────────────────────────────────────────
    criterion = SignatureVerificationLoss(
        num_classes, emb_dim=256, s=arcface_scale, m=arcface_margin,
        margin=contrastive_margin, lambda_arc=lambda_arc, lambda_con=lambda_con,
    ).to(device)

    # ── Optimizer ─────────────────────────────────────────────────────────────
    optimizer = AdamW([
        {"params": model.parameters(),             "lr": lr,       "weight_decay": weight_decay},
        {"params": criterion.arcface.parameters(), "lr": lr * 0.1, "weight_decay": weight_decay},
    ])

    # ── Scheduler ─────────────────────────────────────────────────────────────
    scheduler = OneCycleLR(
        optimizer,
        max_lr=[lr, lr * 0.1],
        total_steps=num_epochs * len(train_loader),
        pct_start=0.1, anneal_strategy="cos",
        div_factor=25, final_div_factor=1e4,
    )

    # ── AMP scaler ────────────────────────────────────────────────────────────
    scaler = GradScaler(enabled=(device.type == "cuda"))

    # ── Training loop ─────────────────────────────────────────────────────────
    best_auc, best_epoch, no_improve = 0.0, 0, 0
    history = {"train_loss": [], "val_loss": [], "val_auc": [], "val_acc": []}

    print("\n" + "="*65)
    print("TRAINING  —  ArcFace + Contrastive  (GPU augmentation)")
    print("="*65)

    for epoch in range(1, num_epochs + 1):
        print(f"\nEpoch {epoch}/{num_epochs}")
        tl = train_epoch(model, criterion, optimizer, scheduler, scaler, train_loader, epoch)
        vl = val_epoch(model, criterion, val_loader)
        vm = evaluate(model, val_loader, "Val")

        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["val_auc"].append(vm["auc"])
        history["val_acc"].append(vm["acc"])

        if vm["auc"] > best_auc:
            best_auc, best_epoch, no_improve = vm["auc"], epoch, 0
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "arc_state": criterion.arcface.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "val_auc": best_auc, "threshold": vm["threshold"],
                "author2id": author2id,
            }, os.path.join(save_dir, "best_model.pt"))
            print(f"  ✓ Best saved  AUC={best_auc:.4f}")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  ⏹ Early stop at epoch {epoch}")
                break

    # ── Test ──────────────────────────────────────────────────────────────────
    print("\n" + "="*65 + "\nTEST EVALUATION\n" + "="*65)
    ckpt = torch.load(os.path.join(save_dir, "best_model.pt"), map_location=device)
    model.load_state_dict(ckpt["model_state"])
    tm = evaluate(model, test_loader, "Test")
    print(f"\nBest epoch={best_epoch}  |  Val AUC={best_auc:.4f}")
    print(f"Test AUC={tm['auc']:.4f}  Acc={tm['acc']:.4f}  FAR={tm['far']:.4f}  FRR={tm['frr']:.4f}")

    # ── Plots ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"],   label="Val")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")
    axes[1].plot(history["val_auc"])
    axes[1].set_title("Val AUC"); axes[1].set_xlabel("Epoch")
    axes[2].plot(history["val_acc"])
    axes[2].set_title("Val Accuracy"); axes[2].set_xlabel("Epoch")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "training_curves.png"), dpi=150)
    plt.close()
    print(f"  Curves → {save_dir}/training_curves.png")

    return model, history, tm


# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def verify_pair(model, img1_tensor, img2_tensor, threshold=0.5):
    model.eval()
    e1, e2 = model(img1_tensor.to(device), img2_tensor.to(device))
    dist   = F.pairwise_distance(e1.float(), e2.float()).item()
    return {
        "distance":   dist,
        "threshold":  threshold,
        "is_genuine": dist < threshold,
        "confidence": float(np.clip(1.0 - dist / (threshold * 2), 0, 1)),
        "verdict":    "GENUINE" if dist < threshold else "FORGED",
    }


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    trained_model, history, test_metrics = train(
        model              = raw_model,
        train_dataloader   = train_dataloader,
        val_dataloader     = val_dataloader,
        test_dataloader    = test_dataloader,
        train_pairs        = train_pairs,
        val_pairs          = val_pairs,
        test_pairs         = test_pairs,
        genuine_by_author  = genuine_by_author,
        num_epochs         = 40,
        lr                 = 3e-4,
        weight_decay       = 1e-4,
        arcface_margin     = 0.50,
        arcface_scale      = 32.0,
        contrastive_margin = 1.0,
        lambda_arc         = 0.6,
        lambda_con         = 0.4,
        save_dir           = "./checkpoints",
        patience           = 10,
    )

Genuine authors: 268, dict_keys(['001', '002', '003', '004', '006', '009', '012', '014', '015', '016', '021', '022', '023', '024', '025', '026', '027', '028', '029', '036', '037', '038', '039', '040', '041', '042', '043', '044', '045', '052', '053', '054', '055', '056', '057', '058', '059', '060', '061', '068', '069', '070', '071', '072', '073', '074', '075', '076', '077', '084', '085', '086', '087', '088', '089', '090', '091', '092', '093', '100', '101', '102', '103', '104', '105', '118', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '170', '171', '172', '173', '174', '230', '231', '233', '234', '235', '238', '240', '241', '243', '246', '249', '250', '251', '315', '318', '320', '328', '333', '341', '

C:\Users\gupta\AppData\Local\Temp\ipykernel_18852\693194934.py:384: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device.type == "cuda"))


  Cached 7327 unique images  (120 MB in RAM)
  Loaders (num_workers=0, batch 128/512): train=471 | val=19 | test=19 batches

TRAINING  —  ArcFace + Contrastive  (GPU augmentation)

Epoch 1/40


Epoch 001 [Train]:   0%|                                                                       | 0/471 [00:00<?, ?it/s]C:\Users\gupta\AppData\Local\Temp\ipykernel_18852\693194934.py:266: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\gupta\AppData\Local\Temp\ipykernel_18852\693194934.py:299: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  [Train] Loss=12.7341  ArcFace=21.0303  Contrastive=0.2898
  [Val  ] Loss=13.4028
  [Val] Acc=0.8218  AUC=0.8974  FAR=0.2026  FRR=0.1538  Thresh=0.3072
  ✓ Best saved  AUC=0.8974

Epoch 2/40


...

Epoch 009 [Train]:   0%|                                                                       | 0/471 [00:00<?, ?it/s]C:\Users\gupta\AppData\Local\Temp\ipykernel_18852\693194934.py:266: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\gupta\AppData\Local\Temp\ipykernel_18852\693194934.py:299: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  [Train] Loss=0.8914  ArcFace=1.3039  Contrastive=0.2725
  [Val  ] Loss=4.2524
  [Val] Acc=0.9194  AUC=0.9777  FAR=0.0677  FRR=0.0934  Thresh=0.7410
  ✓ Best saved  AUC=0.9777

Epoch 10/40


Epoch 010 [Train]:   0%|                                                                       | 0/471 [00:00<?, ?it/s]C:\Users\gupta\AppData\Local\Temp\ipykernel_18852\693194934.py:266: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\gupta\AppData\Local\Temp\ipykernel_18852\693194934.py:299: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  [Train] Loss=0.7483  ArcFace=1.0683  Contrastive=0.2683
  [Val  ] Loss=3.0195
  [Val] Acc=0.9191  AUC=0.9759  FAR=0.0818  FRR=0.0800  Thresh=0.6913

Epoch 11/40


Epoch 011 [Train]:   0%|                                                                       | 0/471 [00:00<?, ?it/s]C:\Users\gupta\AppData\Local\Temp\ipykernel_18852\693194934.py:266: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
Epoch 011 [Train]:   8%|██                       | 40/471 [00:03<00:40, 10.69it/s, arc=1.0788, con=0.2687, loss=0.7548]